# Live Inference Walkthrough

**Use this only if someone in the room specifically asks "can it actually run live on new audio?"** The packaged Docker demo (`demo_app.py`) shows *pre-computed* results for a curated set of clips — instant, offline, no GPU. This notebook instead actually downloads real model weights and runs them, narrated step by step.

What this notebook does NOT do: reproduce the CxER judge classification live. That needs Llama-3.3-70B under vLLM on a multi-GPU host (see `scripts/02_run_cxer_judge.py`) — not something to spin up mid-demo. Instead, the last cell shows the judge's *pre-computed* verdict for the same clip, for context.

**Before running:** `pip install -r requirements.txt python-dotenv` (transformers/jiwer come from requirements.txt; python-dotenv is otherwise only pulled in via requirements-judge.txt). This also needs internet access (to pull model weights from Hugging Face) and, for the fine-tuned checkpoint, a valid `HF_TOKEN` — either exported in your shell or in a `.env` file in this repo's root (same as `scripts/02_run_cxer_judge.py` expects). It also needs real disk space/bandwidth (each Whisper checkpoint is roughly 1.5–3 GB) and will take noticeably longer on CPU than on a GPU.

In [ ]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # picks up HF_TOKEN / HF_USERNAME from .env if present
HF_TOKEN = os.environ.get("HF_TOKEN")
HF_USERNAME = os.environ.get("HF_USERNAME")
print("HF_TOKEN set:", bool(HF_TOKEN), "| HF_USERNAME:", HF_USERNAME)

## 1. Pick a model family and a clip

`FINETUNED_ID` is a best guess (`<your HF username>/<checkpoint name>`, matching the short names in `faa_cxer.config.DEFAULT_MODEL_PAIRS`) — double check it against whatever your actual Hugging Face repo id is before relying on it.

`AUDIO_PATH` defaults to one of the same clips already bundled for the packaged demo (`data/audio_samples/`), so the live output can be compared directly against what's shown there — point it at any other `.wav` to transcribe something new.

In [ ]:
# --- edit these for the family / clip you want to demo ---
PRETRAINED_ID = "openai/whisper-large-v3"
FINETUNED_ID = f"{HF_USERNAME}/FAA_Whisper_L_TL_500K"  # verify this matches your actual HF repo id

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AUDIO_PATH = REPO_ROOT / "data" / "audio_samples" / "5075.wav"  # the "rambling hallucination, fixed" Whisper Large clip
CURATED_JSON = REPO_ROOT / "data" / "curated_samples.json"

print("Audio file:", AUDIO_PATH, "exists:", AUDIO_PATH.exists())

## 2. Load both models

This is the slow part on CPU — downloading + loading each checkpoint typically takes 1–3 minutes the first time (cached afterward).

In [ ]:
from transformers import pipeline

def load_asr_pipeline(model_id: str):
    print(f"Loading {model_id} ...")
    t0 = time.time()
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model_id,
        token=HF_TOKEN,
        device=-1,  # CPU; set to 0 if you have a GPU available
    )
    print(f"  loaded in {time.time() - t0:.1f}s")
    return pipe

pretrained_pipe = load_asr_pipeline(PRETRAINED_ID)
finetuned_pipe = load_asr_pipeline(FINETUNED_ID)

## 3. Run both models on the clip, live

In [ ]:
import IPython.display as ipd

ipd.display(ipd.Audio(str(AUDIO_PATH)))

def transcribe(pipe, audio_path):
    t0 = time.time()
    result = pipe(str(audio_path))
    elapsed = time.time() - t0
    return result["text"].strip(), elapsed

pretrained_text, pretrained_time = transcribe(pretrained_pipe, AUDIO_PATH)
finetuned_text, finetuned_time = transcribe(finetuned_pipe, AUDIO_PATH)

print(f"Pretrained ({PRETRAINED_ID}), {pretrained_time:.2f}s:\n  {pretrained_text}\n")
print(f"Fine-tuned ({FINETUNED_ID}), {finetuned_time:.2f}s:\n  {finetuned_text}\n")

## 4. Compare against the reference + the pre-computed judge verdict

WER is computed here live (via `jiwer`, same normalization the pipeline uses). The CxER classification is NOT re-run — it's pulled from `data/curated_samples.json`, i.e. whatever the LLM judge decided the last time this exact model was evaluated on this exact clip. If your live transcription differs from the model's stored prediction (different decoding settings, a model update on the Hub, etc.), say so explicitly rather than implying the judge saw *this* run's output.

In [ ]:
import jiwer

def normalize(text):
    transform = jiwer.Compose([jiwer.ToLowerCase(), jiwer.RemovePunctuation(), jiwer.RemoveMultipleSpaces(), jiwer.Strip()])
    return transform([text])[0]

curated = json.loads(CURATED_JSON.read_text())
matching = [s for s in curated["samples"] if s["audio_file"] == AUDIO_PATH.name]
reference = matching[0]["reference"] if matching else input("No stored reference for this clip — type it in: ")
ref_norm = normalize(reference)

print("Reference:", reference)
print(f"Pretrained WER (live):  {jiwer.wer(ref_norm, normalize(pretrained_text)):.3f}")
print(f"Fine-tuned WER (live):  {jiwer.wer(ref_norm, normalize(finetuned_text)):.3f}")

if matching:
    print(f"\nJudge model: {curated['judge']} (pre-computed, not re-run here)")
    for variant in matching[0]["variants"]:
        print(f"  {variant['label']}: {variant['classification']} — {variant['explanation']}")
else:
    print("\n(No stored judge verdict for this clip — it wasn't one of the curated samples.)")